In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


## Environment Setup, Secrets & W&B Initialization

In [6]:
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import wandb
from kaggle_secrets import UserSecretsClient

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(seed=42)

user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WanDB_API_KEY")
wandb.login(key=wandb_api_key)

wandb.init(
    project="DL-23f1000967-notebook-t22026",
    name="model3_roberta_base",
    config={
        "model_name": "roberta-base",
        "max_length": 256,
        "batch_size": 8,
        "grad_accum_steps": 2,
        "epochs": 3,
        "learning_rate": 2e-5,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "n_folds": 5
    }
)

config = wandb.config
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23f1000967 (23f1000967-indian-institute-of-technology-madras) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Using device: cuda


## Load competition Data

In [7]:
train_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv')
OPTS = ['A', 'B', 'C', 'D', 'E']

encoder = LabelEncoder()
train_df['label'] = encoder.fit_transform(train_df['answer'])

print(f"Train Dataset Shape: {train_df.shape}")
print(f"Test Dataset Shape:  {test_df.shape}\n")

Train Dataset Shape: (2000, 9)
Test Dataset Shape:  (500, 7)



In [8]:
tokenizer = AutoTokenizer.from_pretrained(config.model_name)

class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_length, has_labels=True):
        self.prompts = df['prompt'].fillna("").tolist()
        self.options = df[OPTS].fillna("").values.tolist()
        self.labels = df['label'].tolist() if has_labels else None
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        prompt = self.prompts[idx]
        opts = self.options[idx]
        encoded = self.tokenizer(
            [prompt] * 5, opts,
            truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt'
        )
        item = {k: v for k, v in encoded.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

## MAP@3 Metric

In [9]:
def map_at_3(probs, y_true):
    top3 = np.argsort(-probs, axis=1)[:, :3]
    score = 0.0
    for i in range(len(y_true)):
        ranks = list(top3[i])
        if y_true[i] in ranks:
            score += 1.0 / (ranks.index(y_true[i]) + 1)
    return score / len(y_true)

In [10]:
skf = StratifiedKFold(n_splits=config.n_folds, shuffle=True, random_state=42)

oof_probs = np.zeros((len(train_df), 5))
test_probs_sum = np.zeros((len(test_df), 5))
fold_scores = []

test_dataset = MCQDataset(test_df, tokenizer, config.max_length, has_labels=False)
test_loader = DataLoader(test_dataset, batch_size=config.batch_size, shuffle=False)

for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df['label'])):
    train_fold = train_df.iloc[tr_idx].reset_index(drop=True)
    val_fold = train_df.iloc[va_idx].reset_index(drop=True)

    train_dataset = MCQDataset(train_fold, tokenizer, config.max_length)
    val_dataset = MCQDataset(val_fold, tokenizer, config.max_length)
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False)

    model = AutoModelForMultipleChoice.from_pretrained(config.model_name, torch_dtype=torch.float32).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=config.weight_decay)
    total_steps = (len(train_loader) // config.grad_accum_steps) * config.epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=int(total_steps * config.warmup_ratio), num_training_steps=total_steps)
    scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

    for epoch in range(1, config.epochs + 1):
        model.train()
        running_loss = 0.0
        optimizer.zero_grad()
        for step, batch in enumerate(train_loader):
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                outputs = model(**batch)
                loss = outputs.loss / config.grad_accum_steps
            scaler.scale(loss).backward()
            running_loss += loss.item() * config.grad_accum_steps

            if (step + 1) % config.grad_accum_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

        epoch_loss = running_loss / len(train_loader)
        wandb.log({"fold": fold, "epoch": epoch, "train_loss": epoch_loss})
        print(f"Fold {fold} | Epoch {epoch}/{config.epochs} | train loss: {epoch_loss:.4f}")

    model.eval()
    val_probs = []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits = model(**batch).logits
            val_probs.append(torch.softmax(logits, dim=1).float().cpu().numpy())
    val_probs = np.concatenate(val_probs, axis=0)
    fold_map3 = map_at_3(val_probs, val_fold['label'].values)
    oof_probs[va_idx] = val_probs
    fold_scores.append(fold_map3)
    wandb.log({"fold": fold, "val_map3": fold_map3})
    print(f"Fold {fold} | val MAP@3: {fold_map3:.4f}")

    test_probs = []
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                logits = model(**batch).logits
            test_probs.append(torch.softmax(logits, dim=1).float().cpu().numpy())
    test_probs_sum += np.concatenate(test_probs, axis=0)

    del model
    torch.cuda.empty_cache()

oof_map3 = map_at_3(oof_probs, train_df['label'].values)
wandb.log({"mean_fold_map3": np.mean(fold_scores), "oof_map3": oof_map3})
wandb.finish()
print(f"Mean fold MAP@3: {np.mean(fold_scores):.4f}")
print(f"Overall OOF MAP@3: {oof_map3:.4f}")

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForMultipleChoice LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
classifier.weight               | MISSING    | 
roberta.pooler.dense.weight     | MISSING    | 
classifier.bias                 | MISSING    | 
roberta.pooler.dense.bias       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
/tmp/ipykernel_58/481927202.py:24: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.

Fold 0 | Epoch 1/3 | train loss: 1.5190
Fold 0 | Epoch 2/3 | train loss: 0.8771
Fold 0 | Epoch 3/3 | train loss: 0.5529


/tmp/ipykernel_58/481927202.py:55: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):


... (24 outputs left out)

Fold 4 | val MAP@3: 0.9762


/tmp/ipykernel_58/481927202.py:69: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):


epoch,▁▅█▁▅█▁▅█▁▅█▁▅█
fold,▁▁▁▁▃▃▃▃▅▅▅▅▆▆▆▆████
mean_fold_map3,▁
oof_map3,▁
train_loss,█▃▁▇▃▁█▄▂█▅▂█▃▁
val_map3,▅▆▁▅█
epoch,3
fold,4
mean_fold_map3,0.93675
oof_map3,0.93675
train_loss,0.49799


Mean fold MAP@3: 0.9367
Overall OOF MAP@3: 0.9367


In [11]:
test_probs = test_probs_sum / config.n_folds
classes = np.array(OPTS)
map_predictions = []

for row_probs in test_probs:
    top_3_indices = np.argsort(row_probs)[::-1][:3]
    top_3_labels = classes[top_3_indices]
    map_predictions.append(" ".join(top_3_labels))

submission_df = pd.DataFrame({
    'id': test_df['id'],
    'prediction': map_predictions
})
submission_df.to_csv('submission.csv', index=False)
display(submission_df.head(5))

,id,prediction
0,1,A D B
1,2,B E A
2,3,B E D
3,4,E C A
4,5,C A D
